# 10-Node Random — 2nd-Order Additive

Parameter sweep over (P x K) space with stochastic noise (Euler-Maruyama).

**Metrics computed:** Gaussian TC/DTC, cumulant k123, powercorr, skewness, kurtosis, KSG TC/DTC/O-info.
**Signals saved:** Raw E and I time series per parameter point.


In [1]:
import sys, os
import numpy as np
sys.path.append('../')

from Scripts.parameters_10nodes import (
    N_nodes, param_nodes, M_rand, T_ho_rand,
    dt, sigma_E, P_values, K_values, K3_values,
)
from Scripts.simulation_10nodes import simulate_wc_stochastic_10n
from Scripts.oscillation_detection import detect_limit_cycle_poincare
from Scripts.metrics import (
    entropy_gauss, TC_gauss, DTC_gauss, cumulants, dev_gauss, powercorr,
    clean_rowwise_signed, TC_ksg, DTC_ksg, Oinfo_ksg,
)

# Topology for this notebook
M = M_rand
T_ho = T_ho_rand

print(f'N_nodes = {N_nodes}')
print(f'M degrees: {M.sum(axis=1).astype(int).tolist()}')
print('Imports OK.')


[10-node] N = 10
  Ring-SW:  degrees = [5, 4, 5, 4, 4, 5, 4, 5, 4, 4], mean = 4.4, edges = 22
  Random :  degrees = [3, 7, 5, 3, 4, 2, 6, 3, 1, 2], mean = 3.6, edges = 18
[10-node] thetaE perturbations: [-0.011, -0.148, 0.29, 0.03, 0.282, -0.165, 0.24, -0.106, -0.102, 0.128]
[10-node] thetaI perturbations: [0.253, 0.469, 0.455, 0.431, 0.137, -0.1, -0.36, -0.113, -0.185, 0.111]
N_nodes = 10
M degrees: [3, 7, 5, 3, 4, 2, 6, 3, 1, 2]
Imports OK.


## Stochastic Parameter Sweep (P x K)

20x20 grid. For each point we compute Gaussian metrics, KSG metrics,
and save the raw (E, I) time series.


In [3]:
# =====================================================================
# Parameter sweep with stochastic noise
# =====================================================================

# Matrices to store metrics
oscillation_map_noise = np.zeros((len(P_values), len(K_values)), dtype=int)
TC_map_noise = np.zeros((len(P_values), len(K_values)))
DTC_map_noise = np.zeros((len(P_values), len(K_values)))
Cumulant_map_noise = np.zeros((len(P_values), len(K_values)))
PowerCorr_map_noise = np.zeros((len(P_values), len(K_values)))
Entropy_map_noise = np.zeros((len(P_values), len(K_values)))
Entropy_map_skew = np.zeros((len(P_values), len(K_values)))
Entropy_map_kurt = np.zeros((len(P_values), len(K_values)))
TC_ksg_map = np.zeros((len(P_values), len(K_values)))
DTC_ksg_map = np.zeros((len(P_values), len(K_values)))
osc_fraction_map = np.zeros((len(P_values), len(K_values)))
Oinfo_ksg_map = np.zeros((len(P_values), len(K_values)))

T_sim = 375  # simulation time (s)

# Fixed initial state for reproducibility
seed = 42
rng = np.random.default_rng(seed)
state0 = 0.1 * rng.standard_normal(2 * N_nodes)

# Folder for raw signals
sim_folder = '../simulations/10n_random_2nd_additive'
os.makedirs(sim_folder, exist_ok=True)

for i, P_val in enumerate(P_values):
    for j, K_val in enumerate(K_values):
        P_vec = np.full(N_nodes, P_val)
        t, states = simulate_wc_stochastic_10n(state0, P_vec, K_val, M, T_sim, dt, sigma_E=sigma_E)

        start_idx = int(0.2 * len(t))
        E_node = states[start_idx:, :N_nodes]
        I_node = states[start_idx:, N_nodes:]

        # Oscillation detection on ALL nodes (majority criterion: >50%)
        n_osc = 0
        for node_idx in range(N_nodes):
            is_lc_node, _, _, _ = detect_limit_cycle_poincare(
                E_node[:, node_idx], I_node[:, node_idx], dt,
                threshold_ratio=0.05, period_cv_threshold=0.05)
            n_osc += int(is_lc_node)
        oscillation_map_noise[i, j] = int(n_osc > N_nodes // 2)  # 1 if majority oscillates
        osc_fraction_map[i, j] = n_osc / N_nodes  # fraction of oscillating nodes

        # Data matrix (excitatory signals only)
        E = E_node.copy()
        X = E.copy()

        # Covariance matrix
        Sigma = np.cov(X.T)

        # --- Gaussian metrics ---
        TC_map_noise[i, j] = TC_gauss(Sigma)
        DTC_map_noise[i, j] = DTC_gauss(Sigma)
        Entropy_map_noise[i, j] = entropy_gauss(Sigma)
        skew, kurt = dev_gauss(X[:, :3])  # first 3 nodes for compatibility
        Entropy_map_skew[i, j] = skew
        Entropy_map_kurt[i, j] = kurt
        if X.shape[1] >= 3:
            Cumulant_map_noise[i, j] = cumulants(X[:, :3])
            PowerCorr_map_noise[i, j] = powercorr(X[:, :3])

        # --- KSG non-parametric metrics ---
        TC_ksg_map[i, j] = TC_ksg(X)
        DTC_ksg_map[i, j] = DTC_ksg(X)
        Oinfo_ksg_map[i, j] = Oinfo_ksg(X)

        # --- Save raw signals ---

    if (i + 1) % 5 == 0:
        print(f'Progress: {i+1}/{len(P_values)}')

# =====================================================================
# Clean metrics
# =====================================================================
TC_clean = clean_rowwise_signed(TC_map_noise, n_std=2)
DTC_clean = clean_rowwise_signed(DTC_map_noise, n_std=2)
Cumulant_clean = clean_rowwise_signed(Cumulant_map_noise, n_std=2)
PowerCorr_clean = clean_rowwise_signed(PowerCorr_map_noise, n_std=2)
Skew_clean = clean_rowwise_signed(Entropy_map_skew, n_std=2)
Kurt_clean = clean_rowwise_signed(Entropy_map_kurt, n_std=2)
TC_ksg_clean = clean_rowwise_signed(TC_ksg_map, n_std=2)
DTC_ksg_clean = clean_rowwise_signed(DTC_ksg_map, n_std=2)
Oinfo_ksg_clean = clean_rowwise_signed(Oinfo_ksg_map, n_std=2)

# =====================================================================
# Save cleaned metrics
# =====================================================================
np.savez_compressed(
    '../results/metrics_10n_random_pairwise_noise.npz',
    oscillation_map=oscillation_map_noise,
    TC=TC_clean, DTC=DTC_clean,
    cumulant=Cumulant_clean, powercorr=PowerCorr_clean,
    skew=Skew_clean, kurt=Kurt_clean,
    TC_ksg=TC_ksg_clean, DTC_ksg=DTC_ksg_clean, Oinfo_ksg=Oinfo_ksg_clean,
    osc_fraction=osc_fraction_map,
    P=P_values, K3=K_values
)

# Save raw (uncleaned) metrics
np.savez_compressed(
    '../results/raw_metrics_10n_random_pairwise_noise.npz',
    oscillation_map=oscillation_map_noise,
    TC=TC_map_noise, DTC=DTC_map_noise,
    cumulant=Cumulant_map_noise, powercorr=PowerCorr_map_noise,
    skew=Entropy_map_skew, kurt=Entropy_map_kurt,
    TC_ksg=TC_ksg_map, DTC_ksg=DTC_ksg_map, Oinfo_ksg=Oinfo_ksg_map,
    osc_fraction=osc_fraction_map,
    P=P_values, K3=K_values
)

print('Done! Metrics and signals saved.')


Progress: 5/20
Progress: 10/20
Progress: 15/20
Progress: 20/20
Done! Metrics and signals saved.


## Visualization


In [4]:
import plotly.graph_objects as go

def plot_metrics_contour(matrix, P_vals, K_vals, title='Metric Map',
                         colorscale='Inferno', n_contours=15):
    z_min = np.nanmin(matrix)
    z_max = np.nanmax(matrix)
    fig = go.Figure(data=go.Contour(
        z=matrix, x=K_vals, y=P_vals,
        colorscale=colorscale,
        contours=dict(start=z_min, end=z_max,
                      size=(z_max - z_min) / n_contours,
                      coloring='heatmap', showlines=True),
        colorbar=dict(thickness=12, len=0.8)
    ))
    fig.update_layout(title=title, xaxis_title='K',
                      yaxis_title='P', width=500, height=400,
                      margin=dict(l=60, r=40, t=50, b=50))
    fig.show()

# Load and plot
data = np.load('../results/metrics_10n_random_pairwise_noise.npz')
metrics_to_plot = ['oscillation_map', 'TC', 'DTC', 'cumulant', 'powercorr',
                   'skew', 'kurt', 'TC_ksg', 'DTC_ksg', 'Oinfo_ksg', 'osc_fraction']
for m in metrics_to_plot:
    plot_metrics_contour(data[m], data['P'], data['K3'], title=m)


## Network Topology


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

G = nx.from_numpy_array(M)
pos = nx.circular_layout(G)

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
degrees = dict(G.degree())
node_sizes = [300 + 100 * degrees[n] for n in G.nodes()]
nx.draw(G, pos, ax=ax, with_labels=True, node_color='steelblue',
        node_size=node_sizes, font_color='white', font_weight='bold',
        edge_color='gray', width=1.5)
ax.set_title('10-Node Random — 2nd-Order Additive — Topology')
plt.tight_layout()
plt.show()

print(f'Degrees: {[degrees[n] for n in sorted(G.nodes())]}')
print(f'Edges: {G.number_of_edges()}')
